# Silver — `estornos`

## Bibliotecas

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

## Variáveis

In [0]:
# origem
bronze_estornos   = "novarota.bronze.estornos"
silver_transacoes = "novarota.silver.transacoes"   # pra conferir se a transação existe e marcar as estornadas

# destino
tabela_silver     = "novarota.silver.estornos"
tabela_quarentena = "novarota.silver.quarentena_estornos"
tabela_controle   = "novarota.silver.controle_cargas"

# chave negocio
chaves_negocio = ["id_estorno"]

print(f"Tabela alvo: {tabela_silver}")

## Ultima carga

In [0]:
# cargas que existem na bronze
cargas_bronze = [l.data_referencia for l in spark.table(bronze_estornos).select("data_referencia").distinct().collect()]

# cargas que a silver de estornos já processou
if spark.catalog.tableExists(tabela_controle):
    cargas_processadas = [
        l.data_referencia
        for l in spark.table(tabela_controle).filter("tabela = 'estornos'").select("data_referencia").distinct().collect()
    ]
else:
    cargas_processadas = []

cargas_pendentes = sorted(c for c in cargas_bronze if c not in cargas_processadas)

print(f"Cargas já processadas: {sorted(cargas_processadas)}")
print(f"Cargas pendentes: {cargas_pendentes}")

if not cargas_pendentes:
    dbutils.notebook.exit("Nada pendente na bronze de estornos")

## Transformações

In [0]:
filtro_incremental = "where data_referencia in (" + ", ".join(f"'{c}'" for c in cargas_pendentes) + ")"

query = f"""
with estornos as (select *, to_json(struct(*)) as registro_original
from {bronze_estornos} {filtro_incremental})
,
validado as (
select
    a.id_estorno,
    a.id_transacao,
    try_cast(a.data_estorno as date) as data_estorno,
    a.motivo_estorno,
    a.data_referencia,
    a.timestamp_ingestao,
    case
        when a.id_estorno is null then 'CHAVE_NULA'
        when try_cast(a.data_estorno as date) is null then 'DATA_INVALIDA'
        when b.id_transacao is null then 'TRANSACAO_INEXISTENTE'
    end as motivo_erro,
    a.registro_original
from estornos a
left join {silver_transacoes} b on a.id_transacao = b.id_transacao)


select
    *,
    row_number() over (
        partition by id_estorno
        order by case when motivo_erro is null then 0 else 1 end,
                 data_referencia desc,
                 timestamp_ingestao desc
    ) as rn
from validado
"""

df = spark.sql(query)

print(f"Linhas lidas da origem (cargas pendentes): {df.count()}")

## Válidos x quarentena

In [0]:
# válidos: sem motivo de erro e uma linha por estorno
df_validos = (
    df.filter("motivo_erro is null and rn = 1")
      .drop("motivo_erro", "registro_original", "rn")
)

# quarentena: tudo que tem motivo de erro, com a linha como chegou da bronze
df_quarentena = (
    df.filter("motivo_erro is not null")
      .select(
          F.col("id_estorno").alias("chave"),
          "motivo_erro",
          "registro_original",
          "data_referencia",
          F.current_timestamp().alias("timestamp_quarentena"),
      )
)

print(f"Válidos (após dedup): {df_validos.count()}")
print(f"Quarentena: {df_quarentena.count()}")

## Conferência

In [0]:
dups_no_lote = (
    df_validos.groupBy(*chaves_negocio).count()
    .filter(F.col("count") > 1)
    .count()
)
print(f"Chaves duplicadas no lote atual (esperado 0): {dups_no_lote}")

df_quarentena.display()

## Carga incremental — MERGE

In [0]:
if not spark.catalog.tableExists(tabela_silver):
    (
        df_validos.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(tabela_silver)
    )
    print(f"Tabela criada (carga inicial): {tabela_silver} | {df_validos.count()} linhas")
else:
    tabela = DeltaTable.forName(spark, tabela_silver)
    condicao = " AND ".join([f"t.{k} = s.{k}" for k in chaves_negocio])

    (
        tabela.alias("t")
        .merge(df_validos.alias("s"), condicao)
        # fato: estorno não muda. Se já existe, ignora
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"MERGE concluído em {tabela_silver}")

## Marca as transações estornadas

In [0]:
# olha todos os estornos da silver, não só os da carga: assim um estorno que chega semanas depois
# também marca a transação antiga, so muda a fl_estornada
spark.sql(f"""
merge into {silver_transacoes} t
using (select distinct id_transacao from {tabela_silver}) e
on t.id_transacao = e.id_transacao
when matched and t.fl_estornada = false then
    update set t.fl_estornada = true
""")

spark.sql(f"""
select fl_estornada, count(*) as qtd, sum(valor) as valor
from {silver_transacoes}
group by fl_estornada
""").display()

## Quarentena e controle

In [0]:
cargas_sql = ", ".join(f"'{c}'" for c in cargas_pendentes)

# quarentena: antes de gravar, apaga o que já tiver dessas cargas. Se a célula rodar de novo
# (ou o job fizer retry depois de uma falha), regrava o mesmo conteúdo em vez de duplicar
if spark.catalog.tableExists(tabela_quarentena):
    spark.sql(f"delete from {tabela_quarentena} where data_referencia in ({cargas_sql})")
df_quarentena.write.format("delta").mode("append").saveAsTable(tabela_quarentena)

# controle: uma linha por carga processada. Fica por último, assim a carga só conta como processada se tudo antes deu certo
controle = (
    df.groupBy("data_referencia")
      .agg(
          F.sum(F.when(F.col("motivo_erro").isNull(), 1).otherwise(0)).alias("linhas_validas"),
          F.sum(F.when(F.col("motivo_erro").isNotNull(), 1).otherwise(0)).alias("linhas_quarentena"),
      )
      .select(
          F.lit("estornos").alias("tabela"),
          "data_referencia",
          F.col("linhas_validas").cast("bigint"),
          F.col("linhas_quarentena").cast("bigint"),
          F.current_timestamp().alias("processado_em"),
      )
)
if spark.catalog.tableExists(tabela_controle):
    spark.sql(f"delete from {tabela_controle} where tabela = 'estornos' and data_referencia in ({cargas_sql})")
controle.write.format("delta").mode("append").saveAsTable(tabela_controle)

controle.display()

## Conferência pós-escrita

In [0]:
print(f"Conferência da tabela: {tabela_silver}")

df_silver = spark.table(tabela_silver)
print(f"Total de linhas: {df_silver.count()}")

dups = (
    df_silver.groupBy(*chaves_negocio).count()
    .filter(F.col("count") > 1)
    .count()
)
print(f"Chaves duplicadas (esperado 0): {dups}")

df_silver.limit(50).display()